In [34]:
import os
import json
import pandas as pd

In [35]:
model_names_d = {
    # "BAAI/bge-small-en-v1.5": "bge-s",
    # "BAAI/bge-small-zh-v1.5": "bge-s",
    "BAAI/bge-base-en-v1.5": "bge-b",
    "BAAI/bge-base-zh-v1.5": "bge-b",
    "Alibaba-NLP/gte-multilingual-base": "gte",
    "moka-ai/m3e-base": "m3e",
    # "google-bert/bert-base-multilingual-uncased": "bert-ml",
    # "answerdotai/ModernBERT-base": "modern"
}

methods = [
    "FC",
    # "FCDualLabelAwareAffine",
    # "FCLabelAwareAffine",
    "FCLabelAware",
    "FCFusion",
    # "FCSimpleLabelAware",
    # "FCDualLabelAware",
    "MultiHead",
    # "MultiHeadSimpleLabelAware",
    "MultiHeadLabelAware",
    "MultiHeadFusion",
    # "MultiHeadDualLabelAware",
    # "Try_MultiHeadLabelAware",
    # "FCLabelAwareAffineSimple",
    # "MultiHeadAffine",
    # "MultiHeadAffine_Simple",
    # "MultiHeadLabelAwareAffineSimple"
    # "MultiHeadLabelAwareAffine",
    # "MultiHeadDualLabelAwareAffine",
    # "DualMultiHeadLabelAwareAffine",
    # "DualMultiHeadLabelAwareAffineSimple",
    # "MultiHeadDualLabelAwareAffineV2",
]

dataset_names_d = {
    "amazon_reviews": "amazon",
    "DBPedia": "DBPedia",
    "industry_middle2sub": "industry",
    "web-of-science": "wos",
}

In [36]:
output_dir = "/mnt/disk/model_output/llm_industry_distillation_paper/output/"

In [37]:
def find_largest_checkpoint(folder):
    checkpoint_max_num = 0
    if not os.path.exists(folder):
        return None

    for fold in os.listdir(folder):
        if fold.startswith('checkpoint-'):
            try:
                # 提取checkpoint-后的数字部分并转换为整数
                checkpoint_num = int(fold.split('checkpoint-')[1])
                # 比较并更新最大数字和对应的文件夹名
                if checkpoint_num > checkpoint_max_num:
                    checkpoint_max_num = checkpoint_num
            except ValueError:
                print(f"警告：文件夹 {fold} 包含非数字的编号，已跳过")
                continue

    if checkpoint_max_num == 0:
        return None

    return os.path.join(folder, 'checkpoint-{}'.format(checkpoint_max_num))

In [38]:
def get_column_name(model_name, dataset_name):
    short_model = model_names_d[model_name]
    short_dataset = dataset_names_d[dataset_name]
    return f"{short_dataset}-{short_model}"

In [39]:
def get_result(middle_dir="PN/batch32"):
    # index 是方法，columns 是dataset
    ans = []
    for method in methods:
        # 遍历所有数据集
        method_d = {}
        for dataset_name in dataset_names_d.keys():
            for model_name in model_names_d.keys():
                # cur_model_dir
                if "Fusion" in method:
                    cur_middle_dir = "PN/batch32e3"
                else:
                    cur_middle_dir = middle_dir
                save_model_dir = os.path.join(
                    output_dir,
                    dataset_name,
                    method,
                    model_name,
                    cur_middle_dir,
                )
                largest_checkpoint = find_largest_checkpoint(save_model_dir)

                if largest_checkpoint is None:
                    continue
                trainer_state_json_file = os.path.join(largest_checkpoint, 'trainer_state.json')
                column = get_column_name(dataset_name=dataset_name, model_name=model_name)
                data = json.load(open(trainer_state_json_file))
                best_metric = round(data['best_metric'] * 100, 2)
                method_d[column] = best_metric
        ans.append(method_d)
    return ans

In [7]:
methods

['FC', 'FCLabelAware', 'FCFusion', 'MultiHead', 'MultiHeadLabelAware']

In [43]:
result_df = pd.DataFrame(
    get_result("PN/batch32e10"),
    index=methods
)
result_df

,amazon-bge-b,amazon-gte,amazon-m3e,DBPedia-bge-b,DBPedia-gte,DBPedia-m3e,industry-bge-b,industry-gte,industry-m3e,wos-bge-b,wos-gte,wos-m3e
FC,76.64,77.92,72.24,97.84,97.57,97.09,79.63,79.82,79.55,87.95,88.09,84.39
FCLabelAware,76.96,77.65,72.37,97.74,98.01,97.10,79.31,80.45,79.33,88.08,88.58,83.72
FCFusion,78.55,NaN,NaN,97.90,NaN,NaN,NaN,NaN,NaN,89.03,NaN,NaN
MultiHead,78.50,79.13,74.67,97.80,97.64,97.34,79.22,79.96,79.06,87.91,88.46,84.73
MultiHeadLabelAware,77.64,77.87,72.79,97.82,97.76,97.04,78.75,79.97,78.87,87.90,88.22,83.57
MultiHeadFusion,79.40,80.24,75.68,98.02,97.98,97.18,79.87,80.86,80.19,88.78,89.07,85.24


In [33]:
result_df = pd.DataFrame(
    get_result("PN/batch32e10"),
    index=methods
)
result_df

,amazon-bge-b,amazon-gte,amazon-m3e,DBPedia-bge-b,DBPedia-gte,DBPedia-m3e,industry-bge-b,industry-gte,industry-m3e,wos-bge-b,wos-gte,wos-m3e
FC,76.64,77.92,72.24,97.84,97.57,97.09,79.63,79.82,79.55,87.95,88.09,84.39
FCLabelAware,76.96,77.65,72.37,97.74,98.01,97.10,79.31,80.45,79.33,88.08,88.58,83.72
FCFusion,78.52,79.18,73.85,97.92,97.82,97.34,80.63,80.99,80.55,89.07,89.20,85.08
MultiHead,78.50,79.13,74.67,97.80,97.64,97.34,79.22,79.96,79.06,87.91,88.46,84.73
MultiHeadLabelAware,77.64,77.87,72.79,97.82,97.76,97.04,78.75,79.97,78.87,87.90,88.22,83.57
MultiHeadFusion,79.42,80.13,75.65,98.00,97.92,97.14,79.90,80.80,80.17,88.73,88.99,85.22


In [1]:
result_df = pd.DataFrame(
    get_result(),
    index=methods
)
result_df

NameError: name 'pd' is not defined

In [10]:
result_df = pd.DataFrame(
    get_result(),
    index=methods
)
result_df

,amazon-bge-s,amazon-bge-b,amazon-bert-ml,DBPedia-bge-s,DBPedia-bge-b,DBPedia-bert-ml,wos-bge-s,wos-bge-b,wos-bert-ml
FC,65.06,74.84,71.43,97.50,97.75,97.72,87.00,87.56,86.52
FCDualLabelAwareAffine,76.47,77.19,73.06,97.69,97.77,97.55,87.85,87.94,86.38
FCLabelAwareAffine,76.04,77.25,73.89,97.68,97.73,97.55,87.72,88.22,86.07
MultiHeadAffine,76.94,78.74,75.51,97.62,97.86,97.72,87.82,87.69,87.05
MultiHeadLabelAwareAffine,77.23,77.28,74.05,97.64,97.90,97.47,87.86,88.01,86.32
MultiHeadDualLabelAwareAffine,77.47,77.53,74.20,97.58,97.76,97.74,88.07,87.91,86.37
DualMultiHeadLabelAwareAffine,76.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultiHeadDualLabelAwareAffineV2,77.16,77.85,NaN,97.41,97.76,NaN,NaN,NaN,NaN


In [ ]:
result_df = pd.DataFrame(
    get_result(),
    index=methods
)
result_df